In [34]:
from alerce.core import Alerce
import pandas as pd
import pyvo as vo
import requests
import sqlalchemy as sa
import sys
import numpy as np
from astropy.coordinates import SkyCoord
from astropy import units as u

In [35]:
alerce = Alerce()
alerce_tap = vo.dal.TAPService('https://tap.alerce.online/tap')

In [13]:
ztf_objects = alerce.query_objects(
    survey="ztf",
    classifier="lc_classifier",
    class_name="SN",
    format="pandas"
)
print(ztf_objects)

Empty DataFrame
Columns: []
Index: []


In [36]:
ob = alerce.query_object(oid="ZTF26aacextk", survey="ztf", format="pandas")
ob

,oid,ndethist,ncovhist,mjdstarthist,mjdendhist,corrected,stellar,ndet,g_r_max,g_r_max_corr,g_r_mean,g_r_mean_corr,firstmjd,lastmjd,deltajd,meanra,meandec,sigmara,sigmadec,step_id_corr
0,ZTF26aacextk,23,2217,61058.43963,61078.519271,False,False,6,None,None,None,None,61059.56081,61078.519271,18.958461,187.020241,6.967839,0.030032,0.02981,27.5.7a32.dev1


In [37]:
filename = "../Data/tns_search (1).csv"
sources = pd.read_csv(filename)
sources

,ID,Name,RA,DEC,Obj. Type,Redshift,Host Name,Host Redshift,Reporting Group/s,Discovery Data Source/s,...,Discovery Mag/Flux,Discovery Filter,Discovery Date (UT),Sender,Remarks,Unreal,Discovery Bibcode,Classification Bibcodes,Ext. catalog/s,Auto classification
0,207943,SN 2026loz,13:26:37.234,-12:06:11.00,SN Ia,0.141000,NaN,NaN,"GOTO, ZTF","GOTO, ZTF",...,19.9000,L-GOTO,2026-05-07 14:51:45.792,GOTO_bot,NaN,NaN,2026TNSTR1968....1O,2026TNSCR1999....1H,NaN,NaN
1,207922,SN 2026log,15:38:25.317,+02:50:28.66,SN Ia,0.076221,LEDA 1238879,0.07622,"ALeRCE, GOTO, ATLAS","ZTF, GOTO, ATLAS",...,18.7549,g-ZTF,2026-05-07 08:51:56.998,ALeRCE,NaN,NaN,2026TNSTR1964....1M,2026TNSCR2000....1F,NaN,NaN
2,207908,SN 2026lns,13:29:49.691,-05:45:05.96,SN Ia,0.068600,NaN,NaN,"ZTF, GOTO, LAST, ATLAS","ZTF, GOTO, LAST, ATLAS",...,19.2662,r-ZTF,2026-05-07 05:03:27.996,Fritz,NaN,NaN,2026TNSTR1975....1S,NaN,NaN,NaN
3,207886,SN 2026lmw,13:07:35.674,+15:13:24.73,SN Ia,0.077618,NaN,NaN,"ZTF, ATLAS","ZTF, ATLAS",...,19.2447,g-ZTF,2026-05-02 06:00:17.003,Fritz,NaN,NaN,2026TNSTR1975....1S,2026TNSCR2000....1F,NaN,NaN
4,207830,SN 2026lky,12:45:09.372,-35:13:48.39,SN Ia,0.056000,NaN,NaN,"GOTO, ATLAS","GOTO, ATLAS",...,18.7600,L-GOTO,2026-05-05 10:22:43.680,GOTO_bot,NaN,NaN,2026TNSTR1940....1O,2026TNSCR1962....1K,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,206493,SN 2026jqy,17:37:46.128,-00:10:27.29,SN IIn,0.045000,SDSS J173746.12-001028.4,NaN,"ALeRCE, Fink, ATLAS, GOTO","ZTF, ATLAS, GOTO",...,19.9423,r-ZTF,2026-04-15 10:33:17.001,ALeRCE,NaN,NaN,2026TNSTR1627....1M,2026TNSCR1886....1S,NaN,NaN
96,206490,SN 2026jqv,14:18:06.700,+18:27:11.82,SN Ia-pec,0.050000,NaN,NaN,"ATLAS, WFST","ATLAS, WFST",...,18.5930,wide-ATLAS,2026-04-15 02:03:58.176,ATLAS_Bot1,NaN,NaN,2026TNSTR1628....1T,2026TNSCR1816....1S,NaN,NaN
97,206486,SN 2026jqr,15:43:49.397,+30:31:32.48,SN Ia,0.064000,NaN,NaN,"ATLAS, GOTO, ZTF, WFST","ATLAS, GOTO, ZTF, WFST",...,19.3960,wide-ATLAS,2026-04-15 01:42:23.040,ATLAS_Bot1,NaN,NaN,2026TNSTR1628....1T,2026TNSCR1903....1J,NaN,NaN
98,206466,SN 2026jqi,21:35:46.821,-63:54:12.77,SN Ic,0.010300,NaN,NaN,ATLAS,ATLAS,...,18.8080,orange-ATLAS,2026-04-15 09:11:30.336,ATLAS_Bot1,NaN,NaN,2026TNSTR1628....1T,"2026TNSCR1708....1F, 2026TNSCR1723....1S",NaN,NaN


In [53]:
ra = np.array(sources["RA"])
dec = np.array(sources["DEC"])
z = np.array(sources["Redshift"])
supernovaes = []
# for r, d in zip(ra, dec):
#     #print(r, d)
#     coord = SkyCoord(ra=r, dec=d, unit=(u.hourangle, u.deg))
#     supernovae = alerce.query_objects(ra=coord.ra.deg, dec=coord.dec.deg, survey="ztf", format="pandas")
#     supernovaes.append(supernovae)
# supernovaes
#print(ra)

In [39]:
coord = SkyCoord(ra=ra[:, None], dec=dec[:,None], unit=(u.hourangle, u.deg))

In [54]:
#print(coord)

In [58]:
super_ra = coord.ra.deg
super_dec = coord.dec.deg
#print(super_ra)
supernovaes = []
for r, d in zip(super_ra, super_dec):
    # print(r, d)
    supernovae = alerce.query_objects(ra=r, dec=d, survey="ztf", format="pandas")
    supernovaes.append(supernovae)

supernovaes

[            oid ndethist  ncovhist  mjdstarthist    mjdendhist  corrected  \
 0  ZTF20abbcsrb        1       190  58992.230521  58992.230521      False   
 1  ZTF20aasibxr       15       659  58912.437465  60458.228958       True   
 2  ZTF25aciozba        1      1067  61026.519375  61026.519375       True   
 3  ZTF26aauvuig        8      1222  61167.231539  61171.317431      False   
 
    stellar  ndet g_r_max g_r_max_corr  ...       lastmjd   deltajd  \
 0    False     1    None         None  ...  58992.230521  0.000000   
 1    False     1    None         None  ...  60458.228958  0.000000   
 2    False     1    None         None  ...  61026.519375  0.000000   
 3    False     8    None         None  ...  61171.317431  4.085891   
 
        meanra    meandec   sigmara  sigmadec  class  classifier  probability  \
 0  201.654118 -12.104484       NaN       NaN   None        None         None   
 1  201.653957 -12.102476  0.066478  0.065000   None        None         None   
 2  201.

In [60]:
supernovaes[np.where(supernovaes, None)]

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (100,) + inhomogeneous part.